# Project 7: Evaluation and Tuning

## Decision and primary metric (specified before fitting)

The model supports the bank marketing team in ranking customers for review before potential term-deposit outreach. Review capacity is limited; ranking eventual subscribers early matters more than correctly identifying the many non-subscribers. This is an observational prediction of subscription, **not** an estimate of causal uplift from calling.

**Primary metric: Average Precision (AP).** The team wants to prioritize a limited number of reviews, so a useful ranking must concentrate subscribers near the top instead of maximizing majority-class accuracy. We choose AP before model fitting because it measures precision across recall operating points for the rare positive outcome; a constant-score classifier's AP equals the evaluation period's positive prevalence.

**Secondary metrics:** ROC-AUC (pairwise ranking); log-loss (probability quality); precision at top-K (review efficiency); recall at top-K (fraction of observed subscribers found).

**Evaluation contract:** Chronological 60% training / 20% development validation / 20% untouched final test. Preprocessing is fitted only within a training fold via pipelines; `TimeSeriesSplit` preserves order. The test is first **scored** in the last code cell and is not used to select models or review budgets. Split variables are created earlier without examining their outcomes. The originally selected random forest and top-1,000 budget are locked before the final test evaluation.

**Time/group limits:** Row order is chronological, but exact per-contact timestamps and customer identifiers are unavailable. Repeat customers cannot be reliably isolated across splits. Unknown categorical values are treated as missing within the preprocessing pipeline; call `duration` is excluded because it is unavailable before contact.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data_path = Path("../week07/data/bank-additional-full.csv")

bank_df = pd.read_csv(data_path, sep=";")

print("Dataset shape:", bank_df.shape)

display(bank_df.head())

print("Columns:")
print(bank_df.columns.tolist())

Dataset shape: (41188, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


Columns:
['age', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'duration', 'campaign', 'pdays', 'previous', 'poutcome', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'y']


In [2]:
target = bank_df['y'].map({'no': 0, 'yes': 1})
assert target.notna().all()

#Count positive and egative
positive_count = int(target.sum())
negative_count = int((target == 0).sum())
positive_rate = target.mean()

#Count completely identical rows
duplicate_count = bank_df.duplicated().sum()

#Count actual NaN values
nan_count = bank_df.isna().sum()

unknown_count = bank_df.eq('unknown').sum()

audit_table = pd.DataFrame({
    "NaN values": nan_count,
    "Unknown values": unknown_count
})

print("Total rows:", len(bank_df))
print("Positive outcomes:", positive_count)
print("Negative outcomes:", negative_count)
print("Positive rate:", f"{positive_rate:.2%}")
print("Exact duplicate rows:", duplicate_count)

display(audit_table.sort_values("Unknown values", ascending = False))



Total rows: 41188
Positive outcomes: 4640
Negative outcomes: 36548
Positive rate: 11.27%
Exact duplicate rows: 12


,NaN values,Unknown values
default,0,8597
education,0,1731
housing,0,990
loan,0,990
job,0,330
marital,0,80
age,0,0
poutcome,0,0
nr.employed,0,0
euribor3m,0,0


In [3]:
# Remove the target and post-contact duration
X_all = bank_df.drop(
    columns=["y", "duration"]
)

y_all = target.copy()

# Calculate split boundaries
n = len(bank_df)

train_end = int(n * 0.60)
validation_end = int(n * 0.80)

# Chronological training set
X_train = X_all.iloc[:train_end].copy()
y_train = y_all.iloc[:train_end].copy()

# Chronological validation set
X_val = X_all.iloc[train_end:validation_end].copy()
y_val = y_all.iloc[train_end:validation_end].copy()

# Final test set: leave untouched until final evaluation
X_test = X_all.iloc[validation_end:].copy()
y_test = y_all.iloc[validation_end:].copy()

print("Training rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

print("Training positive rate:", f"{y_train.mean():.2%}")
print("Validation positive rate:", f"{y_val.mean():.2%}")

print("Test set has been reserved. Do not score it yet.")

assert len(X_train) + len(X_val) + len(X_test) == n
assert X_train.shape[1] >= 8

Training rows: 24712
Validation rows: 8238
Test rows: 8238
Training positive rate: 4.81%
Validation positive rate: 11.07%
Test set has been reserved. Do not score it yet.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

numeric_columns = X_train.select_dtypes(include="number").columns.to_list()
categorical_columns = X_train.select_dtypes(exclude="number").columns.to_list()

def build_preprocessor():

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(missing_values="unknown", strategy="most_frequent", fill_value="Missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_columns),
        ("categorical", categorical_pipeline, categorical_columns)
    ])

    return preprocessor

print("Numerical columns:", numeric_columns)
print("Categorical columns:", categorical_columns)

Numerical columns: ['age', 'campaign', 'pdays', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed']
Categorical columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'day_of_week', 'poutcome']


In [5]:
from sklearn.model_selection import TimeSeriesSplit

time_cv = TimeSeriesSplit(n_splits=3)

for fold, (train_idx, val_idx) in enumerate(time_cv.split(X_train), start=1):
    assert train_idx.max() < val_idx.min()

    print(f"Fold {fold}: {len(train_idx)} training rows, {len(val_idx)} validation rows")

    print(f"Positive rates: {y_train.iloc[train_idx].mean():.2%} - {y_train.iloc[val_idx].mean():.2%}")

Fold 1: 6178 training rows, 6178 validation rows
Positive rates: 2.96% - 4.05%
Fold 2: 12356 training rows, 6178 validation rows
Positive rates: 3.50% - 6.39%
Fold 3: 18534 training rows, 6178 validation rows
Positive rates: 4.47% - 5.83%


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier


logistic_pipeline = Pipeline([
    ("pre", build_preprocessor()),
    ("model", LogisticRegression(solver="lbfgs", max_iter=3000, random_state=42))
])


forest_pipeline = Pipeline([
    ("pre", build_preprocessor()),
    ("model", RandomForestClassifier(random_state=42, n_jobs=1))
])

print("Both model pipelines defined.")

Both model pipelines defined.


In [7]:
from scipy.stats import loguniform, randint
from sklearn.model_selection import RandomizedSearchCV


logistic_parameters = {
    "model__C": loguniform(1e-3, 1e2),
    "model__class_weight": [None, "balanced"],
    "pre__categorical__imputer__strategy": ["most_frequent", "constant"]
}


forest_parameters = {
    "model__n_estimators": randint(60, 121),
    "model__max_depth": [5, 10, 15, None],
    "model__min_samples_leaf": loguniform(0.001, 0.03),
    "model__max_features": ["sqrt", 0.5, 1.0],
    "pre__categorical__imputer__strategy": ["most_frequent", "constant"]
}


print("Search spaces defined.")

Search spaces defined.


In [8]:
logistic_search = RandomizedSearchCV(
    estimator=logistic_pipeline,
    param_distributions=logistic_parameters,
    n_iter=20,
    scoring="average_precision",
    cv=time_cv,
    n_jobs=2,
    random_state=42,
    refit=True,
    error_score="raise",
    return_train_score=True,
    verbose=1
)

forest_search = RandomizedSearchCV(
    estimator=forest_pipeline,
    param_distributions=forest_parameters,
    n_iter=20,
    scoring="average_precision",
    cv=time_cv,
    n_jobs=2,
    random_state=42,
    refit=True,
    error_score="raise",
    return_train_score=True,
    verbose=1
)

# Only the chronological training partition is used here.
logistic_search.fit(X_train, y_train)

forest_search.fit(X_train, y_train)

print("Logistic search completed.")
print("Random forest search completed.")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Fitting 3 folds for each of 20 candidates, totalling 60 fits
Logistic search completed.
Random forest search completed.


In [9]:
def make_search_table(search):

    results = pd.DataFrame(search.cv_results_)

    parameter_columns = [
        column for column in results.columns if column.startswith("param_")
    ]

    fold_columns = [f"split{i}_test_score"
        for i in range(time_cv.n_splits)
    ]

    selected_columns = (
        ["rank_test_score", "mean_test_score",
         "std_test_score", "mean_train_score"]
        + fold_columns
        + parameter_columns
    )

    table = results[selected_columns].copy()

    return table.sort_values(
        "rank_test_score"
    ).reset_index(drop=True)


logistic_results = make_search_table(
    logistic_search
)

forest_results = make_search_table(
    forest_search
)

print("LOGISTIC REGRESSION: ALL 20 CONFIGURATIONS")
display(logistic_results.round(4))

print("RANDOM FOREST: ALL 20 CONFIGURATIONS")
display(forest_results.round(4))

LOGISTIC REGRESSION: ALL 20 CONFIGURATIONS


,rank_test_score,mean_test_score,std_test_score,mean_train_score,split0_test_score,split1_test_score,split2_test_score,param_model__C,param_model__class_weight,param_pre__categorical__imputer__strategy
0,1,0.0533,0.0113,0.0565,0.0396,0.0672,0.0531,67.3225,balanced,constant
1,2,0.0533,0.0131,0.0604,0.0397,0.0709,0.0491,2.6373,NaN,constant
2,3,0.0531,0.0124,0.0553,0.0394,0.0695,0.0505,0.0060,NaN,most_frequent
3,4,0.0531,0.0122,0.0573,0.0395,0.0692,0.0507,0.0100,NaN,constant
4,5,0.0531,0.0113,0.0564,0.0396,0.0672,0.0526,14.5282,balanced,constant
5,6,0.0530,0.0126,0.0557,0.0394,0.0697,0.0500,0.0081,NaN,most_frequent
6,7,0.0530,0.0121,0.0556,0.0395,0.0688,0.0508,0.0013,balanced,constant
7,8,0.0530,0.0113,0.0565,0.0396,0.0672,0.0523,1.1462,balanced,constant
8,9,0.0528,0.0119,0.0584,0.0395,0.0684,0.0504,0.0333,NaN,constant
9,10,0.0528,0.0130,0.0580,0.0396,0.0705,0.0482,4.5706,NaN,most_frequent


RANDOM FOREST: ALL 20 CONFIGURATIONS


,rank_test_score,mean_test_score,std_test_score,mean_train_score,split0_test_score,split1_test_score,split2_test_score,param_model__max_depth,param_model__max_features,param_model__min_samples_leaf,param_model__n_estimators,param_pre__categorical__imputer__strategy
0,1,0.0562,0.0140,0.0763,0.0412,0.0748,0.0525,15,1.0,0.0283,116,most_frequent
1,2,0.0559,0.0127,0.0716,0.0435,0.0734,0.0508,10,sqrt,0.0267,109,constant
2,3,0.0559,0.0125,0.0701,0.0434,0.0730,0.0512,10,sqrt,0.0271,103,constant
3,4,0.0557,0.0146,0.0940,0.0403,0.0753,0.0514,5,1.0,0.0101,68,most_frequent
4,5,0.0556,0.0137,0.1114,0.0418,0.0743,0.0507,15,sqrt,0.0077,83,most_frequent
5,6,0.0552,0.0149,0.1815,0.0401,0.0755,0.0499,None,0.5,0.0060,103,most_frequent
6,7,0.0551,0.0151,0.0996,0.0398,0.0758,0.0498,10,1.0,0.0168,85,most_frequent
7,8,0.0545,0.0147,0.1161,0.0383,0.0739,0.0514,5,1.0,0.0046,82,most_frequent
8,9,0.0544,0.0151,0.1468,0.0383,0.0747,0.0502,15,1.0,0.0080,67,most_frequent
9,10,0.0544,0.0149,0.2796,0.0393,0.0746,0.0493,15,0.5,0.0038,109,constant


In [10]:
def summarize_search(search, family_name):

    results = pd.DataFrame(
        search.cv_results_
    )

    best_row = results.loc[
        results["rank_test_score"] == 1
    ].iloc[0]

    best_mean = best_row["mean_test_score"]
    best_std = best_row["std_test_score"]

    standard_error = (
        best_std / np.sqrt(time_cv.n_splits)
    )

    within_one_se = results[
        results["mean_test_score"]
        >= best_mean - standard_error
    ]

    print(f"\n{family_name}")
    print("Best mean CV AP:", round(best_mean, 4))
    print("Fold standard deviation:", round(best_std, 4))
    print("Approximate standard error:", round(standard_error, 4))
    print("Candidates within 1 SE:", len(within_one_se))
    print("Best parameters:")
    print(search.best_params_)


summarize_search(
    logistic_search,
    "LOGISTIC REGRESSION"
)

summarize_search(
    forest_search,
    "RANDOM FOREST"
)


LOGISTIC REGRESSION
Best mean CV AP: 0.0533
Fold standard deviation: 0.0113
Approximate standard error: 0.0065
Candidates within 1 SE: 20
Best parameters:
{'model__C': np.float64(67.32248920775338), 'model__class_weight': 'balanced', 'pre__categorical__imputer__strategy': 'constant'}

RANDOM FOREST
Best mean CV AP: 0.0562
Fold standard deviation: 0.014
Approximate standard error: 0.0081
Candidates within 1 SE: 20
Best parameters:
{'model__max_depth': 15, 'model__max_features': 1.0, 'model__min_samples_leaf': np.float64(0.028336828367886542), 'model__n_estimators': 116, 'pre__categorical__imputer__strategy': 'most_frequent'}


In [11]:
from sklearn.metrics import average_precision_score

baseline_rows = []

for fold, (train_idx, val_idx) in enumerate(
    time_cv.split(X_train),
    start=1
):
    # Positive rate of this fold's validation rows
    y_fold = y_train.iloc[val_idx]

    # Constant predictions: no customer-specific ranking
    base_rate = y_fold.mean()

    dummy_scores = np.full(
        len(y_fold),
        base_rate
    )

    baseline_ap = average_precision_score(
        y_fold,
        dummy_scores
    )

    # Saved scores from the winning configurations
    logistic_ap = logistic_search.cv_results_[
        f"split{fold - 1}_test_score"
    ][logistic_search.best_index_]

    forest_ap = forest_search.cv_results_[
        f"split{fold - 1}_test_score"
    ][forest_search.best_index_]

    baseline_rows.append({
        "Fold": fold,
        "Baseline AP": baseline_ap,
        "Logistic AP": logistic_ap,
        "Forest AP": forest_ap,
        "Logistic gain": logistic_ap - baseline_ap,
        "Forest gain": forest_ap - baseline_ap
    })

baseline_comparison = pd.DataFrame(baseline_rows)

display(baseline_comparison.round(5))

print("\nMEAN SCORES")
print(
    baseline_comparison.drop(columns="Fold").mean().round(5)
)

,Fold,Baseline AP,Logistic AP,Forest AP,Logistic gain,Forest gain
0,1,0.04047,0.03959,0.04118,-0.00088,0.00072
1,2,0.06394,0.06715,0.07479,0.00322,0.01085
2,3,0.05827,0.05311,0.05250,-0.00516,-0.00578



MEAN SCORES
Baseline AP      0.05422
Logistic AP      0.05328
Forest AP        0.05616
Logistic gain   -0.00094
Forest gain      0.00193
dtype: float64


In [12]:
paired_differences = (
    baseline_comparison["Forest AP"]
    - baseline_comparison["Logistic AP"]
)

mean_difference = paired_differences.mean()

difference_std = paired_differences.std(ddof=1)

print("Forest minus logistic AP, by fold:")
print(paired_differences.round(5).to_list())

print("Mean paired difference:", round(mean_difference, 5))
print("Paired difference SD:", round(difference_std, 5))

if abs(mean_difference) < difference_std:
    print(
        "Difference is smaller than fold-to-fold "
        "variation. No clear winner."
    )
else:
    print(
        "Difference exceeds the paired fold spread. "
        "Investigate further before claiming a winner."
    )

Forest minus logistic AP, by fold:
[0.00159, 0.00763, -0.00061]
Mean paired difference: 0.00287
Paired difference SD: 0.00427
Difference is smaller than fold-to-fold variation. No clear winner.


In [13]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss
)

# Both selected pipelines have already been refitted
# on the chronological training partition.
logistic_val_probs = (
    logistic_search.best_estimator_
    .predict_proba(X_val)[:, 1]
)

forest_val_probs = (
    forest_search.best_estimator_
    .predict_proba(X_val)[:, 1]
)

# Constant-probability baseline fitted from training only
training_prevalence = y_train.mean()

baseline_val_probs = np.full(
    len(y_val),
    training_prevalence
)

validation_rows = []

for name, probabilities in [
    ("Prevalence baseline", baseline_val_probs),
    ("Logistic regression", logistic_val_probs),
    ("Random forest", forest_val_probs)
]:
    validation_rows.append({
        "Model": name,
        "Validation AP": average_precision_score(
            y_val, probabilities
        ),
        "ROC-AUC": roc_auc_score(
            y_val, probabilities
        ),
        "Log-loss": log_loss(
            y_val, probabilities
        )
    })

validation_metrics = pd.DataFrame(validation_rows)

display(validation_metrics.round(5))

print(
    "Validation positive prevalence:",
    f"{y_val.mean():.4f}"
)

print(
    "Training positive prevalence:",
    f"{training_prevalence:.4f}"
)

,Model,Validation AP,ROC-AUC,Log-loss
0,Prevalence baseline,0.11071,0.50000,0.37981
1,Logistic regression,0.09918,0.48891,0.85911
2,Random forest,0.09684,0.47351,0.35077


Validation positive prevalence: 0.1107
Training positive prevalence: 0.0481


In [14]:
def capacity_table(y_true, probabilities, budgets):

    actual = np.asarray(y_true)
    scores = np.asarray(probabilities)

    # Sort customers by descending predicted probability
    ranked_indices = np.argsort(-scores)

    ranked_actual = actual[ranked_indices]

    rows = []

    for budget in budgets:

        # Select only the top-ranked customers
        top_labels = ranked_actual[:budget]

        found = int(top_labels.sum())
        wasted = int(budget - found)

        rows.append({
            "Review budget": budget,
            "Subscribers found": found,
            "Wasted reviews": wasted,
            "Precision at budget": found / budget,
            "Recall at budget": found / actual.sum()
        })

    return pd.DataFrame(rows)


review_budgets = [100, 500, 1000, 2000]

logistic_capacity = capacity_table(
    y_val,
    logistic_val_probs,
    review_budgets
)

forest_capacity = capacity_table(
    y_val,
    forest_val_probs,
    review_budgets
)

print("LOGISTIC REGRESSION")
display(logistic_capacity.round(4))

print("RANDOM FOREST")
display(forest_capacity.round(4))

LOGISTIC REGRESSION


,Review budget,Subscribers found,Wasted reviews,Precision at budget,Recall at budget
0,100,8,92,0.080,0.0088
1,500,23,477,0.046,0.0252
2,1000,48,952,0.048,0.0526
3,2000,102,1898,0.051,0.1118


RANDOM FOREST


,Review budget,Subscribers found,Wasted reviews,Precision at budget,Recall at budget
0,100,3,97,0.030,0.0033
1,500,23,477,0.046,0.0252
2,1000,58,942,0.058,0.0636
3,2000,132,1868,0.066,0.1447


In [15]:
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    log_loss,
    confusion_matrix,
    precision_score,
    recall_score
)

from pathlib import Path

# -------------------------------------------------
# 1. LOCKED POLICY: final forest + top-1000 reviews
# -------------------------------------------------

final_model = forest_search.best_estimator_

primary_budget = 1000
review_budgets = [100, 500, 1000, 2000]

# -------------------------------------------------
# 2. TEST PREDICTIONS: perform once
# -------------------------------------------------

test_probabilities = final_model.predict_proba(
    X_test
)[:, 1]

test_actual = y_test.to_numpy()

# Baseline probabilities known from training only
baseline_probabilities = np.full(
    len(test_actual),
    y_train.mean()
)

# -------------------------------------------------
# 3. REPORT THE PRIMARY AND SECONDARY METRICS
# -------------------------------------------------

test_metrics = pd.DataFrame([
    {
        "Model": "Prevalence baseline",
        "AP": average_precision_score(
            test_actual, baseline_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            test_actual, baseline_probabilities
        ),
        "Log-loss": log_loss(
            test_actual, baseline_probabilities
        )
    },
    {
        "Model": "Random forest",
        "AP": average_precision_score(
            test_actual, test_probabilities
        ),
        "ROC-AUC": roc_auc_score(
            test_actual, test_probabilities
        ),
        "Log-loss": log_loss(
            test_actual, test_probabilities
        )
    }
])

# -------------------------------------------------
# 4. PREDECLARED TOP-1000 OPERATING POINT
# -------------------------------------------------

ranked_indices = np.argsort(
    -test_probabilities,
    kind="stable"
)

test_predictions = np.zeros(
    len(test_actual),
    dtype=int
)

test_predictions[
    ranked_indices[:primary_budget]
] = 1

tn, fp, fn, tp = confusion_matrix(
    test_actual,
    test_predictions,
    labels=[0, 1]
).ravel()

operating_metrics = pd.DataFrame([{
    "Review budget": primary_budget,
    "TN": tn,
    "FP": fp,
    "FN": fn,
    "TP": tp,
    "Precision": precision_score(
        test_actual, test_predictions,
        zero_division=0
    ),
    "Recall": recall_score(
        test_actual, test_predictions,
        zero_division=0
    )
}])

# -------------------------------------------------
# 5. CAPACITY RESULTS
# -------------------------------------------------

test_capacity = capacity_table(
    test_actual,
    test_probabilities,
    review_budgets
)

test_capacity["Expected random subscribers"] = (
    test_capacity["Review budget"]
    * test_actual.mean()
)

# -------------------------------------------------
# 6. SAVE THE LOCKED TEST RESULTS
# -------------------------------------------------

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

test_metrics.to_csv(
    output_dir / "p07_test_metrics.csv",
    index=False
)

operating_metrics.to_csv(
    output_dir / "p07_test_confusion.csv",
    index=False
)

test_capacity.to_csv(
    output_dir / "p07_test_capacity.csv",
    index=False
)

print("FINAL HELD-OUT TEST METRICS")
display(test_metrics.round(5))

print("\nCONFUSION MATRIX: TOP 1,000")
display(operating_metrics.round(5))

print("\nREVIEW CAPACITY TABLE")
display(test_capacity.round(4))

print("\nFinal evaluation completed.")

FINAL HELD-OUT TEST METRICS


,Model,AP,ROC-AUC,Log-loss
0,Prevalence baseline,0.30833,0.50000,0.96986
1,Random forest,0.35047,0.50975,0.89836



CONFUSION MATRIX: TOP 1,000


,Review budget,TN,FP,FN,TP,Precision,Recall
0,1000,5138,560,2100,440,0.44,0.17323



REVIEW CAPACITY TABLE


,Review budget,Subscribers found,Wasted reviews,Precision at budget,Recall at budget,Expected random subscribers
0,100,41,59,0.410,0.0161,30.8327
1,500,223,277,0.446,0.0878,154.1636
2,1000,440,560,0.440,0.1732,308.3273
3,2000,778,1222,0.389,0.3063,616.6545



Final evaluation completed.
